In [ ]:
from dataclasses import dataclass

@dataclass

class MiniMindConfig:
    vocab_size: int = 6400  # 使用前确认与 tokenizer 词表大小一致
    hidden_size: int = 128
    num_hidden_layers: int = 2
    num_attention_heads: int = 4
    intermediate_size: int = 384
    max_seq_len: int = 128
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1e6
    dropout: float = 0.0

    @property
    def head_dim(self):
        return self.hidden_size//self.num_attention_heads

In [ ]:
config=MiniMindConfig()
assert config.hidden_size%config.num_attention_heads==0
assert config.head_dim%2==0

print(config)
print(config.head_dim)

MiniMindConfig(vocab_size=6400, hidden_size=128, num_hidden_layers=2, num_attention_heads=4, intermediate_size=384, max_seq_len=128, rms_norm_eps=1e-06, rope_theta=1000000.0, dropout=0.0)
32


In [ ]:
import torch
from torch import nn

class RMSNorm(nn.Module):#对单个token的整条向量进行归一化
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.weight=nn.Parameter(torch.ones(dim))

    def forward(self,x):
        x_float=x.float()
        mean_square=x_float.square().mean(dim=-1,keepdim=True)
        normalized=x_float*torch.rsqrt(mean_square+self.eps)

        return(normalized*self.weight).to(x.dtype)
     



In [ ]:
x=torch.randn(2,5,config.hidden_size)

norm=RMSNorm(config.hidden_size)
y=norm(x)
print(x)